# Querying Pipeline Data on S3

This notebook shows how to connect to the pipeline's Delta tables stored in S3
(or MinIO) and run SQL queries against them.

## Prerequisites

- Set your S3 credentials in the first cell
- If using MinIO locally, also set `S3_ENDPOINT_URL` and `S3_ALLOW_HTTP`
- The pipeline must have been run at least once so that Delta tables exist

## Table Aliases

The pipeline registers tables by alias, not by full path. Available aliases:

| Alias | Layer | Description |
|-------|-------|-------------|
| `ibkr_snapshot_raw` | Raw | IBKR Flex raw data (encrypted values) |
| `ibkr_snapshot_normalized` | Normalized | IBKR positions with decrypted metadata |
| `trading212_snapshot_raw` | Raw | Trading 212 raw data (encrypted values) |
| `trading212_snapshot_normalized` | Normalized | T212 positions with decrypted metadata |
| `xtb_snapshot_raw` | Raw | XTB raw data |
| `xtb_snapshot_normalized` | Normalized | XTB positions |
| `consolidated_holdings` | Normalized | Cross-broker consolidated holdings |
| `portfolio_allocation_analytics` | Analytics | Portfolio allocation by ticker

## 1. Configure S3 Access

Set your credentials below. For MinIO (local Docker), use the defaults shown.

In [ ]:
import os

# S3 bucket — change to your bucket name
os.environ["S3_BUCKET"] = "pipeline"

# AWS credentials — fill in your values (or use MinIO defaults for local dev)
os.environ["AWS_ACCESS_KEY_ID"] = "minioadmin"
os.environ["AWS_SECRET_ACCESS_KEY"] = "minioadmin"
os.environ["AWS_REGION"] = "us-east-1"

# For MinIO or other S3-compatible stores, uncomment:
# os.environ["S3_ENDPOINT_URL"] = "http://localhost:9000"
# os.environ["S3_ALLOW_HTTP"] = "true"

# Encryption key — needed to decrypt financial values
# Set this to your Fernet key (from `python -m pipeline.run keygen`)
# os.environ["ENCRYPTION_KEY"] = "your-key-here"

# To query demo set demo=true and fill in secrets with _DEMO prefixes
# os.environ["DEMO"] = "true"
# os.environ["S3_BUCKET_DEMO"] = "pipeline-demo"
# os.environ["AWS_ACCESS_KEY_ID_DEMO"] = ""
# os.environ["AWS_SECRET_ACCESS_KEY_DEMO"] = ""
# os.environ["AWS_REGION"] = "eu-west-1"
# os.environ["ENCRYPTION_KEY_DEMO"] = ""

## 2. Connect and List Tables

In [ ]:
import polars as pl
from pipeline.query import list_tables, refresh, get_connection

pl.Config.set_tbl_rows(50)

# Discover available Delta tables
refresh()
tables = list_tables()
print(f"Found {len(tables)} tables:")
for t in tables:
    print(f"  • {t}")

In [ ]:
db = get_connection()

## 3. View Portfolio Allocation

The `portfolio_allocation_analytics` table contains the final output of the
pipeline — percentage allocation per ticker, grouped by broker.

In [ ]:
db.sql("SELECT * FROM portfolio_allocation_analytics").pl()

## 4. Allocation by Currency

Aggregate allocation by security currency to see your portfolio's
currency exposure.

In [ ]:
db.sql("""
    SELECT
        security_currency,
        ROUND(SUM(percentage), 2) AS total_pct,
        COUNT(*) AS num_positions
    FROM portfolio_allocation_analytics
    GROUP BY security_currency
    ORDER BY total_pct DESC
""").pl()

## 5. Allocation by Broker

See how your portfolio is distributed across brokers.

In [ ]:
db.sql("""
    SELECT
        broker,
        ROUND(SUM(percentage), 2) AS total_pct,
        COUNT(*) AS num_positions
    FROM portfolio_allocation_analytics
    GROUP BY broker
    ORDER BY total_pct DESC
""").pl()

## 6. Decrypt Raw Data

Raw tables store financial values as Fernet-encrypted bytes. Use `decrypt_df`
to automatically detect and decrypt encrypted columns.

In [ ]:
from pipeline.query import decrypt_df

pl.Config.set_fmt_str_lengths(200)
pl.Config.set_tbl_width_chars(180)

In [ ]:
# Read raw IBKR data and decrypt the encrypted `value` column
raw_ibkr = db.sql("SELECT * FROM ibkr_snapshot_raw").pl()
decrypted = decrypt_df(raw_ibkr)
decrypted.head(10)

## 7. Normalized Holdings

The `consolidated_holdings` table merges positions from all brokers into
a single view with decrypted values.

In [ ]:
consolidated = db.sql("""
    SELECT ticker, broker, security_currency, value, value_currency
    FROM consolidated_holdings
    ORDER BY value DESC
""").pl()
decrypt_df(consolidated)

## 8. Cross-Broker Holdings

Find tickers that appear in more than one broker — useful for checking
deduplication and allocation accuracy.

In [ ]:
db.sql("""
    SELECT ticker, COUNT(DISTINCT broker) AS broker_count, SUM(percentage) AS total_pct
    FROM portfolio_allocation_analytics
    GROUP BY ticker
    HAVING COUNT(DISTINCT broker) > 1
    ORDER BY total_pct DESC
""").pl()